In [0]:
print("Databricks is working!")

In [0]:
# Verify the file
display(dbutils.fs.ls("s3://stocks-bronze-layer/raw/batch/"))

In [0]:
# Read the CSV
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("s3://stocks-bronze-layer/raw/batch/SP500_Historical_Data.csv")

print("Data loaded successfully!")
# display(df.limit(5))

In [0]:
# Actually check the data
display(df.limit(5))

In [0]:
# Check the row count
print("Number of rows:", df.count())

In [0]:
# First validation — inspect the schema
df.printSchema()

In [0]:
# Check for null values
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
# Check duplicates
duplicate_count = (
    df.groupBy("Ticker", "Date")
      .count()
      .filter(col("count") > 1)
      .count()
)

print("Duplicate Ticker-Date combinations:", duplicate_count)

In [0]:
# Check invalid prices
invalid_prices = df.filter(
    (col("Open") <= 0) |
    (col("High") <= 0) |
    (col("Low") <= 0) |
    (col("Close") <= 0) |
    (col("Adj Close") <= 0) |
    (col("High") < col("Low")) |
    (col("High") < col("Open")) |
    (col("High") < col("Close")) |
    (col("Low") > col("Open")) |
    (col("Low") > col("Close"))
)

print("Invalid price records:", invalid_prices.count())

In [0]:
# Inspect the invalid record
display(invalid_prices)

In [0]:
# Verify the exact rule
display(
    df.filter(
        (col("Ticker") == "HUBB") &
        (col("Date") == "2021-05-05")
    )
)

In [0]:
# let's check which condition triggered
problem_row = df.filter(
    (col("Ticker") == "HUBB") &
    (col("Date") == "2021-05-05")
)

display(
    problem_row.select(
        "Ticker",
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close",
        "Volume",
        (col("Low") > col("Open")).alias("Low > Open"),
        (col("Low") > col("Close")).alias("Low > Close"),
        (col("High") < col("Open")).alias("High < Open"),
        (col("High") < col("Close")).alias("High < Close")
    )
)

In [0]:
# Correct price validation
invalid_prices = df.filter(
    (col("Open") <= 0) |
    (col("High") <= 0) |
    (col("Low") <= 0) |
    (col("Close") <= 0) |
    (col("Adj Close") <= 0) |
    (col("Volume") < 0) |
    (col("High") < col("Low"))
)

print("Invalid price records:", invalid_prices.count())

In [0]:
# Date Validation
from pyspark.sql.functions import to_date, col

df = df.withColumn(
    "Date",
    to_date(col("Date"), "yyyy-MM-dd")
)

df.printSchema()

In [0]:
# check whether any dates failed to convert
print("Invalid dates:", df.filter(col("Date").isNull()).count())

In [0]:
# Basic preprocessing
df = df.toDF(
    "ticker",
    "date",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume"
)

display(df.limit(5))

In [0]:
# Check the cleaned schema
df.printSchema()

In [0]:
# Basic Python profiling
columns = df.columns

print("Columns:", columns)
print("Number of columns:", len(columns))
print("Number of rows:", df.count())

In [0]:
# Write to Silver
silver_path = "s3://stocks-silver-layer/processed/batch/"

df.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Data successfully written to Silver!")

In [0]:
# verify the silver bucket
display(dbutils.fs.ls("s3://stocks-silver-layer/processed/batch/"))

In [0]:
# read from silver
silver_df = spark.read.parquet(
    "s3://stocks-silver-layer/processed/batch/"
)

print("Silver row count:", silver_df.count())

display(silver_df.limit(5))

In [0]:
print(silver_df)